# Cross-Industry Accelerator — Create Ontology
### Auto-Create Fabric IQ Ontology from Industry Package or RDF/OWL

This notebook creates a **Fabric IQ Ontology** item using the `fabricontology` package.
It supports two creation paths:

| Mode | Input | API |
|---|---|---|
| **Package** (default) | `.iq` ontology package file | `generate_definition_from_package()` |
| **RDF/OWL** | `.rdf` / `.owl` file or URL | `generate_definition_from_rdf()` |

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. Auto-resolves Lakehouse, Eventhouse, and Semantic Model item IDs from workspace
3. Generates ontology definition with data bindings
4. Creates the ontology item in Fabric
5. Displays entity types, relationship types, and binding summary

> **Prerequisites:**
> 1. Upload `fabriciq_ontology_accelerator-0.1.0-py3-none-any.whl` to Lakehouse Files
> 2. Upload your `.iq` ontology package (or `.rdf`/`.owl` file) to Lakehouse Files
> 3. Run `02_Load_Lakehouse.ipynb` so tables exist for data bindings
> 4. Run `04_Create_Semantic_Model.ipynb` so the semantic model is available as a source

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════
%run ./00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INSTALL FABRIC IQ ONTOLOGY ACCELERATOR
# ════════════════════════════════════════════════════════════════════════
%pip install /lakehouse/default/Files/fabriciq_ontology_accelerator-0.1.0-py3-none-any.whl --q

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# ONTOLOGY CREATION — CONFIGURATION
# ════════════════════════════════════════════════════════════════════════

# ─── Choose creation mode: "package" or "rdf" ─────────────────────────
ONTOLOGY_MODE = "package"   # "package" → .iq file  |  "rdf" → .rdf/.owl file or URL

# ─── Package mode settings ────────────────────────────────────────────
# Path to the .iq ontology package file in Lakehouse Files
ONTOLOGY_PACKAGE_PATH = f"/lakehouse/default/Files/{INDUSTRY}_ontology_package.iq"

# ─── RDF mode settings ────────────────────────────────────────────────
# Either a local file path or a URL to an RDF/OWL file
RDF_SOURCE = ""  # e.g. "/lakehouse/default/Files/ontology.rdf" or "https://.../*.rdf"
RDF_FORMAT = "xml"  # "xml" for .rdf, "turtle" for .ttl, "n3" for .n3

# ─── Binding schema ──────────────────────────────────────────────────
BINDING_LAKEHOUSE_SCHEMA = "dbo"

print(f"Mode:              {ONTOLOGY_MODE}")
print(f"Ontology Name:     {ONTOLOGY_NAME}")
print(f"Lakehouse:         {LAKEHOUSE_NAME}")
print(f"Eventhouse:        {EVENTHOUSE_NAME}")
if ONTOLOGY_MODE == "package":
    print(f"Package Path:      {ONTOLOGY_PACKAGE_PATH}")
else:
    print(f"RDF Source:        {RDF_SOURCE}")
    print(f"RDF Format:        {RDF_FORMAT}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESOLVE WORKSPACE ITEM IDs
# ════════════════════════════════════════════════════════════════════════

import sempy.fabric as fabric
import json

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
items_df = fabric.list_items()

# Resolve Lakehouse item ID
lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
if lh_matches.empty:
    print(f"⚠ Lakehouse '{LAKEHOUSE_NAME}' not found in workspace. Available:")
    print(items_df[items_df["Type"] == "Lakehouse"][["Display Name", "Id"]].to_string(index=False))
    binding_lakehouse_item_id = ""
else:
    binding_lakehouse_item_id = str(lh_matches.iloc[0].Id)
    print(f"✓ Lakehouse: {LAKEHOUSE_NAME} → {binding_lakehouse_item_id}")

# Resolve Eventhouse item ID (optional)
binding_eventhouse_item_id = ""
if EVENTHOUSE_NAME:
    eh_matches = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == EVENTHOUSE_NAME)]
    if eh_matches.empty:
        print(f"⚠ Eventhouse '{EVENTHOUSE_NAME}' not found. Ontology will be created without Eventhouse bindings.")
    else:
        binding_eventhouse_item_id = str(eh_matches.iloc[0].Id)
        print(f"✓ Eventhouse: {EVENTHOUSE_NAME} → {binding_eventhouse_item_id}")
else:
    print("ℹ No Eventhouse configured — Eventhouse bindings will be skipped.")

print(f"\n✅ Workspace ID: {workspace_id}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GENERATE ONTOLOGY DEFINITION (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

from fabricontology import create_ontology_item, generate_definition_from_package, generate_definition_from_rdf
import requests

if ONTOLOGY_MODE == "package":
    # ─── Package-based creation ─────────────────────────────────────
    print(f"Generating ontology from package: {ONTOLOGY_PACKAGE_PATH}")
    log_audit_event("ONTOLOGY_GENERATE", ONTOLOGY_NAME, f"Mode: package, Path: {ONTOLOGY_PACKAGE_PATH}")

    ontology_definition, entity_types, relationship_types, data_bindings, contextualizations = (
        generate_definition_from_package(
            ontology_package_path=ONTOLOGY_PACKAGE_PATH,
            ontology_name=ONTOLOGY_NAME,
            binding_workspace_id=workspace_id,
            binding_lakehouse_item_id=binding_lakehouse_item_id,
            binding_lakehouse_schema_name=BINDING_LAKEHOUSE_SCHEMA,
            binding_eventhouse_item_id=binding_eventhouse_item_id,
            binding_eventhouse_cluster_uri=EVENTHOUSE_CLUSTER_URI,
            binding_eventhouse_database_name=EVENTHOUSE_DATABASE,
        )
    )
    print(f"\n✅ Package definition generated.")
    print(f"   Entity types:        {len(entity_types)}")
    print(f"   Relationship types:  {len(relationship_types)}")
    print(f"   Data bindings:       {len(data_bindings)}")
    print(f"   Contextualizations:  {len(contextualizations)}")

elif ONTOLOGY_MODE == "rdf":
    # ─── RDF/OWL-based creation (ZT: URL validated) ─────────────────
    if RDF_SOURCE.startswith("http://") or RDF_SOURCE.startswith("https://"):
        # ZT: Validate URL against allowlist before fetching
        validate_url(RDF_SOURCE)
        log_audit_event("RDF_URL_VALIDATED", RDF_SOURCE, "URL passed allowlist check")

        print(f"Fetching RDF from URL: {RDF_SOURCE}")
        resp = requests.get(RDF_SOURCE, timeout=30)
        resp.encoding = "utf-8"
        rdf_content = resp.text
        log_audit_event("RDF_FETCHED", RDF_SOURCE, f"{len(rdf_content)} bytes downloaded")
    else:
        print(f"Reading RDF from file: {RDF_SOURCE}")
        log_audit_event("RDF_FILE_READ", RDF_SOURCE, "Reading from local file")
        with open(RDF_SOURCE, "r", encoding="utf-8") as f:
            rdf_content = f.read()

    ontology_definition, entity_types, relationship_types = (
        generate_definition_from_rdf(
            rdf_content=rdf_content,
            rdf_format=RDF_FORMAT,
            ontology_name=ONTOLOGY_NAME,
        )
    )
    data_bindings = []
    contextualizations = []
    print(f"\n✅ RDF definition generated.")
    print(f"   Entity types:        {len(entity_types)}")
    print(f"   Relationship types:  {len(relationship_types)}")
    print(f"   (RDF mode — no data bindings; add them manually in Fabric IQ)")

else:
    raise ValueError(f"Unknown ONTOLOGY_MODE: '{ONTOLOGY_MODE}'. Use 'package' or 'rdf'.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE ONTOLOGY ITEM IN FABRIC (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

print(f"Creating ontology '{ONTOLOGY_NAME}' in workspace...")
log_audit_event("ONTOLOGY_CREATE", ONTOLOGY_NAME, f"Workspace: {workspace_id}")

response = create_ontology_item(
    workspace_id=workspace_id,
    access_token=access_token,
    ontology_item_name=ONTOLOGY_NAME,
    ontology_definition=ontology_definition,
)

if response.status_code in (200, 201, 202):
    print(f"\n✅ Ontology created successfully!")
    print(f"   Name:     {ONTOLOGY_NAME}")
    print(f"   Status:   {response.status_code}")
    resp_json = response.json()
    if "id" in resp_json:
        print(f"   Item ID:  {resp_json['id']}")
    log_audit_event("ONTOLOGY_CREATED", ONTOLOGY_NAME, f"Status: {response.status_code}")
else:
    print(f"\n⚠ Ontology creation returned status {response.status_code}")
    print(f"   Response: {response.text}")
    log_audit_event("ONTOLOGY_CREATE_FAILED", ONTOLOGY_NAME,
        f"Status: {response.status_code}", "ERROR")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# ONTOLOGY SUMMARY
# ════════════════════════════════════════════════════════════════════════

import pandas as pd

print(f"\n{'═'*60}")
print(f"ONTOLOGY SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*60}")

print(f"\n📦 Entity Types ({len(entity_types)}):")
for et in sorted(entity_types):
    print(f"   • {et}")

print(f"\n🔗 Relationship Types ({len(relationship_types)}):")
for rt in sorted(relationship_types):
    print(f"   • {rt}")

if data_bindings:
    print(f"\n📊 Data Bindings ({len(data_bindings)}):")
    for db in data_bindings:
        print(f"   • {db}")

if contextualizations:
    print(f"\n🔍 Contextualizations ({len(contextualizations)}):")
    for c in contextualizations:
        print(f"   • {c}")

print(f"\n✅ Ontology creation complete.")
print(f"   Next: Run 06_Create_Data_Agent.ipynb to create QA and Operations agents.")